# Databricks Lakehouse — Exploration Notebook

This notebook demonstrates the medallion architecture on NYC Taxi TLC data.

**Layers:**
- Bronze — raw ingestion, append-only, schema preserved
- Silver — cleaned, typed, validated, quarantine for invalid rows
- Gold — business aggregates: daily stats, hourly demand, payment mix

Run locally with `deltalake` Python library, or on Databricks Community Edition with PySpark Delta.

---

In [ ]:
import sys
from pathlib import Path

# Adjust this path when running on Databricks
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np

from src import config as cfg
from src.pipeline import LakehousePipeline
from src.utils import read_delta

print('Setup complete')

## 1. Run the Full Pipeline

In [ ]:
# Point at sample data for a quick run
# Change to cfg.RAW_DATA_DIR to use the full 3-month dataset
sample_dir = PROJECT_ROOT / 'data' / 'sample'

pipeline = LakehousePipeline(raw_dir=sample_dir)
summary = pipeline.run()
summary

## 2. Explore Bronze Layer

In [ ]:
bronze_df = read_delta(cfg.DELTA_DIR / cfg.BRONZE_TABLE)
print(f'Bronze rows: {len(bronze_df)}')
print(f'Columns: {list(bronze_df.columns)}')
bronze_df.head()

In [ ]:
# Verify metadata columns
bronze_df[['_source_file', '_ingested_at', '_batch_id']].value_counts()

## 3. Explore Silver Layer

In [ ]:
silver_df = read_delta(cfg.DELTA_DIR / cfg.SILVER_TABLE)
quarantine_df = read_delta(cfg.DELTA_DIR / cfg.SILVER_QUARANTINE_TABLE)

print(f'Silver (valid) rows  : {len(silver_df)}')
print(f'Quarantine rows      : {len(quarantine_df)}')
print(f'Row invariant check  : {len(silver_df) + len(quarantine_df)} == {len(bronze_df)} (bronze)')
silver_df.head()

In [ ]:
# Quarantine reasons
if not quarantine_df.empty:
    print('Quarantine reason distribution:')
    print(quarantine_df['_quarantine_reason'].value_counts())
quarantine_df[['pickup_at', 'fare_amount', 'trip_distance', '_quarantine_reason']].head()

In [ ]:
# Derived column distributions
silver_df[['trip_duration_min', 'fare_per_mile', 'fare_per_minute']].describe()

## 4. Explore Gold Layer

In [ ]:
daily_df = read_delta(cfg.DELTA_DIR / cfg.GOLD_DAILY_TABLE)
print(f'gold_daily_stats: {len(daily_df)} rows')
daily_df.sort_values('total_revenue', ascending=False).head(10)

In [ ]:
hourly_df = read_delta(cfg.DELTA_DIR / cfg.GOLD_HOURLY_TABLE)
print(f'gold_hourly_demand: {len(hourly_df)} rows')

# Peak hour analysis
peak_hours = hourly_df.groupby('hour')['trip_count'].sum().sort_values(ascending=False)
peak_hours.head()

In [ ]:
payment_df = read_delta(cfg.DELTA_DIR / cfg.GOLD_PAYMENT_TABLE)
print(f'gold_payment_mix: {len(payment_df)} rows')

# Payment type distribution
payment_summary = payment_df.groupby('payment_type').agg(
    total_trips=('trip_count', 'sum'),
    avg_pct=('pct_of_daily_trips', 'mean')
).sort_values('total_trips', ascending=False)
payment_summary

## 5. Data Quality Results

In [ ]:
import json

quality_log_dir = cfg.QUALITY_LOG_DIR
log_files = sorted(quality_log_dir.glob('*.json'))
print(f'Quality log files: {len(log_files)}')

for log_file in log_files:
    with open(log_file) as f:
        log = json.load(f)
    status = 'PASS' if log['passed'] else 'FAIL'
    print(f'  [{status}] {log["source_table"]} → {log["target_table"]}')
    for r in log['results']:
        print(f'       [{r["status"].upper():4s}] {r["check_name"]}: {r["message"]}')

## 6. Schema Evolution

In [ ]:
schema_dir = cfg.SCHEMA_DIR
snapshot_files = sorted(schema_dir.glob('*.json'))
print(f'Schema snapshots: {len(snapshot_files)}')

for snap_file in snapshot_files:
    with open(snap_file) as f:
        snap = json.load(f)
    print(f'  {snap["table_name"]}: {len(snap["schema"])} columns (snapshot at {snap["snapshot_at"][:10]})')

evolution_log = schema_dir / 'schema_evolution_log.jsonl'
if evolution_log.exists():
    with open(evolution_log) as f:
        lines = f.readlines()
    print(f'\nSchema changes logged: {len(lines)}')
else:
    print('\nNo schema changes detected in this run.')